In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    PointStruct
)

from fastembed import TextEmbedding

In [3]:
qql_file_path = "../Data/healthcare_conversations.qql"

with open(qql_file_path, "r", encoding="utf-8") as file:

    qql_script = file.read()

print(qql_script[:2000])


INSERT INTO healthcare_conversations
TEXT "
    Patient Query:
    I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!
    
    Doctor Response:
    Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo BPPV, a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are common. The condition

In [4]:
entries = qql_script.split("INSERT INTO healthcare_conversations")

entries = [e.strip() for e in entries if e.strip()]

print("Total QQL Entries:", len(entries))

Total QQL Entries: 111722


In [5]:
import re
import json
parsed_documents = []

for entry in entries:

    # Extract TEXT
    text_match = re.search(
        r'TEXT\s+"(.*?)"\s+METADATA',
        entry,
        re.DOTALL
    )

    # Extract METADATA
    metadata_match = re.search(
        r'METADATA\s+({.*})',
        entry,
        re.DOTALL
    )

    if text_match and metadata_match:

        retrieval_text = text_match.group(1)

        metadata_json = metadata_match.group(1)

        metadata = json.loads(metadata_json)

        parsed_documents.append({
            "retrieval_text": retrieval_text,
            "metadata": metadata
        })

print("Parsed Documents:", len(parsed_documents))

Parsed Documents: 111722


In [6]:
parsed_documents[0]

{'retrieval_text': '\n    Patient Query:\n    I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!\n    \n    Doctor Response:\n    Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo BPPV, a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are common. The condition is due to problem 

In [7]:
embedding_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [8]:
sample_embedding = list(
    embedding_model.embed(
        [parsed_documents[0]["retrieval_text"]]
    )
)[0]

print("Embedding Dimension:", len(sample_embedding))

Embedding Dimension: 384


In [11]:
from qdrant_client import QdrantClient

client = QdrantClient(
    host="localhost",
    port=6333
)

print("Connected to Qdrant!")

Connected to Qdrant!


In [12]:
from qdrant_client.models import VectorParams, Distance

client.recreate_collection(

    collection_name="healthcare_conversations",

    vectors_config=VectorParams(
        size=len(embeddings[0]),
        distance=Distance.COSINE
    )
)

print("Collection created successfully!")

Collection created successfully!


In [13]:
parsed_documents = parsed_documents[:20]

texts = [
    doc["retrieval_text"]
    for doc in parsed_documents
]

embeddings = []

batch_size = 2

for i in range(0, len(texts), batch_size):

    batch = texts[i:i + batch_size]

    batch_embeddings = list(
        embedding_model.embed(batch)
    )

    embeddings.extend(batch_embeddings)

    print(f"Processed batch {(i // batch_size) + 1}")

print("Embeddings generated:", len(embeddings))

Processed batch 1
Processed batch 2
Processed batch 3
Processed batch 4
Processed batch 5
Processed batch 6
Processed batch 7
Processed batch 8
Processed batch 9
Processed batch 10
Embeddings generated: 20


In [14]:
points = []

for idx, doc in enumerate(parsed_documents):
    point = PointStruct(
        id=str(uuid.uuid4()),
        vector=embeddings[idx].tolist(),
        payload={
            "retrieval_text": doc["retrieval_text"],
            "patient_query": doc["metadata"]["patient_query"],
            "doctor_response": doc["metadata"]["doctor_response"],
            "query_tokens": doc["metadata"]["query_tokens"],
            "response_tokens": doc["metadata"]["response_tokens"]
        }
    )
    points.append(point)

print("Points Prepared:", len(points))

Points Prepared: 20


In [15]:
client.upsert(
    collection_name="healthcare_conversations",
    points=points
)

print("Vectors uploaded successfully!")

Vectors uploaded successfully!


In [16]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='healthcare_conversations')])

In [17]:
query = "I have severe chest pain and dizziness"

In [18]:
query_embedding = list(
    embedding_model.embed([query])
)[0]

In [19]:
results = client.query_points(

    collection_name="healthcare_conversations",

    query=query_embedding.tolist(),

    limit=3
)

In [20]:
for result in results.points:

    print("=" * 80)

    print("Score:", result.score)

    print("\nPatient Query:")
    print(result.payload["patient_query"])

    print("\nDoctor Response:")
    print(result.payload["doctor_response"])

Score: 0.6813636

Patient Query:
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!

Doctor Response:
Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo BPPV, a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are common. The condition is due to problem in the ear, and improves in 